# Train ESP32-S3 CSI model - 6 classes (Kaggle)

Notebook này dùng **dữ liệu CSI do bạn thu bằng `csi_labeler`** (ESP32-S3, 384 int8 = 192 cặp I/Q) để train model 6 lớp:

`empty, walk, sit, stand, lie, fall`

## Cách dùng

1. Zip thư mục `D:\dacn2\work\data` (gồm `index.csv` + `raw/...`).
2. Kaggle -> **Datasets -> New Dataset** -> upload zip, đặt tên `esp32s3-csi-labeled`.
3. Trong notebook: **Add Input** -> chọn dataset vừa upload.
4. Nếu cần, sửa `DATA_DIR` ở cell cấu hình bên dưới.
5. **Run All**.
6. Tải file `esp32s3_csi_model.zip` ở `/kaggle/working/` và gửi lại.

## Lưu ý quan trọng

- Notebook **không dùng** model UT-HAR cũ trong `D:\dacn2\train`.
- Dataset hiện tại mới có 5 trial, 1 người, 1 phòng, **chưa có `fall`**. Notebook sẽ cảnh báo và đánh dấu `deployable=false`; kết quả chỉ để kiểm tra pipeline.
- Muốn model dùng được: thu thêm `fall`, tối thiểu 5 lần/nhãn, 2 người, 2 phòng.
- Split ưu tiên theo **trial/người/phòng**. Nếu không đủ trial, notebook dùng split theo thời gian và cảnh báo kết quả lạc quan.


In [ ]:
import os, json, glob, math, random, zipfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---------------------------------------------------------------------------
# CONFIG - sửa ở đây nếu cần
# ---------------------------------------------------------------------------
DATA_DIR = "/kaggle/input/esp32s3-csi-labeled"
OUT_DIR = Path("/kaggle/working/esp32s3_csi_model")

WINDOW = 250          # số gói / cửa sổ (~3.2 s ở 77 gói/s)
STRIDE = 125          # 50% overlap
LABELS = ["empty", "walk", "sit", "stand", "lie", "fall"]

USE_PCA = False       # True: PCA 192 -> 10 (nhẹ cho MCU); False: giữ 192 biên độ
PCA_COMPONENTS = 10

MODEL_VARIANT = "cnn_small"   # "cnn_small" (TFLite Micro) hoặc "cnn_lstm" (PC)
TFLITE_INT8 = True

MIN_WINDOWS_PER_CLASS = 5

print("TensorFlow:", tf.__version__)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)


In [ ]:
# ---------------------------------------------------------------------------
# Tìm index.csv / data root
# ---------------------------------------------------------------------------
def find_index(root: Path):
    root = Path(root)
    if root.is_file() and root.name == "index.csv":
        return root
    cands = sorted(root.rglob("index.csv"))
    if not cands:
        return None
    # Ưu tiên file nằm nông nhất
    return min(cands, key=lambda p: len(p.parts))


data_root = Path(DATA_DIR)
if not data_root.exists():
    print("DATA_DIR không tồn tại, thử auto-detect trong /kaggle/input ...")
    cands = sorted(Path("/kaggle/input").rglob("index.csv"))
    if not cands:
        raise SystemExit(
            "Không tìm thấy index.csv. Hãy upload thư mục data (có index.csv + raw/) "
            "thành Kaggle Dataset và Add Input."
        )
    data_root = cands[0].parent

index_csv = find_index(data_root)
if index_csv is None:
    print("Không thấy index.csv, sẽ quét raw/**/*.csv")
else:
    data_root = index_csv.parent
    print("index.csv:", index_csv)

print("data_root:", data_root)


In [ ]:
# ---------------------------------------------------------------------------
# Load danh sách trial
# ---------------------------------------------------------------------------
def load_trials(root: Path):
    root = Path(root)
    idx = find_index(root)
    trials = []

    if idx is not None:
        df = pd.read_csv(idx)
        for _, row in df.iterrows():
            label = str(row.get("label", "")).strip().lower()
            if not label:
                continue
            packets = pd.to_numeric(row.get("packets", 0), errors="coerce")
            if pd.notna(packets) and float(packets) <= 0:
                continue

            subject = str(row.get("subject", "unknown")).strip()
            room = str(row.get("room", "unknown")).strip()
            trial_id = str(row.get("trial_id", "")).strip()
            p = Path(str(row.get("csv_path", "")))

            if not p.exists():
                p = idx.parent / "raw" / subject / room / label / (trial_id + ".csv")
            if not p.exists() and trial_id:
                matches = sorted(idx.parent.rglob(trial_id + ".csv"))
                if matches:
                    p = matches[0]
            if not p.exists():
                print("  [skip] không thấy file:", row.get("trial_id"))
                continue

            duration = pd.to_numeric(row.get("duration_s", np.nan), errors="coerce")
            try:
                duration = float(duration)
            except Exception:
                duration = float("nan")
            pps = float(packets) / duration if duration and duration > 0 else float("nan")

            trials.append({
                "path": p,
                "trial_id": trial_id or p.stem,
                "label": label,
                "subject": subject,
                "room": room,
                "packets": int(packets) if pd.notna(packets) else None,
                "pps": pps,
            })
    else:
        for p in sorted(root.rglob("raw/**/*.csv")):
            label = p.parent.name.strip().lower()
            room = p.parent.parent.name
            subject = p.parent.parent.parent.name
            trials.append({
                "path": p, "trial_id": p.stem, "label": label,
                "subject": subject, "room": room, "packets": None, "pps": float("nan"),
            })

    return trials


trials = load_trials(data_root)
print("Số trial:", len(trials))
if not trials:
    raise SystemExit("Không có trial nào. Kiểm tra lại cấu trúc data/index.csv + raw/.")

summary = pd.DataFrame([{k: v for k, v in t.items() if k != "path"} for t in trials])
print(summary.to_string(index=False))
print("\nĐếm theo nhãn:")
print(summary["label"].value_counts().to_string())


In [ ]:
# ---------------------------------------------------------------------------
# Parse CSI CSV -> biên độ 192 subcarrier
# csi_json: [Imag1, Real1, Imag2, Real2, ...] (đúng thứ tự esp-csi)
# ---------------------------------------------------------------------------
def parse_csi_csv(path: Path):
    df = pd.read_csv(path, dtype={"csi_json": str})
    if "csi_json" not in df.columns:
        print("  [skip] thiếu cột csi_json:", path.name)
        return None

    amps = []
    bad = 0
    for s in df["csi_json"].values:
        if not isinstance(s, str) or not s.strip():
            bad += 1
            continue
        try:
            arr = np.asarray(json.loads(s), dtype=np.int32)
        except Exception:
            bad += 1
            continue
        if arr.size < 2:
            bad += 1
            continue
        n = arr.size // 2
        imag = arr[0:2 * n:2].astype(np.float32)
        real = arr[1:2 * n:2].astype(np.float32)
        amps.append(np.sqrt(imag * imag + real * real))

    if not amps:
        print("  [skip] không parse được packet:", path.name)
        return None

    lengths = np.array([a.size for a in amps], dtype=np.int32)
    target = int(np.median(lengths))
    if target % 2 != 0:
        target -= 1
    target = max(2, target)

    out = np.zeros((len(amps), target), dtype=np.float32)
    for i, a in enumerate(amps):
        m = min(target, a.size)
        out[i, :m] = a[:m]

    return out, float(np.median(lengths)), bad


trial_data = []
for t in trials:
    res = parse_csi_csv(t["path"])
    if res is None:
        continue
    amp, med_len, bad = res
    t2 = dict(t)
    t2["amp"] = amp
    t2["median_subcarriers"] = med_len
    t2["bad_packets"] = bad
    trial_data.append(t2)

print("\nĐã parse:", len(trial_data), "/", len(trials), "trial")
for t in trial_data:
    print(f"  {t['trial_id']}: label={t['label']:6s} packets={t['amp'].shape[0]:5d} "
          f"subcarriers={t['amp'].shape[1]} bad={t['bad_packets']}")

subcarrier_sizes = sorted(set(t["amp"].shape[1] for t in trial_data))
print("\nKích thước subcarrier:", subcarrier_sizes)
if len(subcarrier_sizes) > 1:
    print("CẢNH BÁO: các trial có số subcarrier khác nhau; đã pad/truncate về median.")
if subcarrier_sizes != [192]:
    print("CẢNH BÁO: model deploy cho firmware đang giả định 192 subcarrier (CSI len=384).")


In [ ]:
# ---------------------------------------------------------------------------
# Cắt cửa sổ + split
# ---------------------------------------------------------------------------
def make_windows(amp, window, stride):
    if amp is None or amp.shape[0] < window:
        return np.empty((0, window, amp.shape[1] if amp is not None else 0), dtype=np.float32)
    starts = range(0, amp.shape[0] - window + 1, stride)
    return np.stack([amp[s:s + window] for s in starts]).astype(np.float32)


X_list, y_list, group_list, trial_list = [], [], [], []
present_labels = [l for l in LABELS if l in {t["label"] for t in trial_data}]

for ti, t in enumerate(trial_data):
    if t["label"] not in present_labels:
        continue
    w = make_windows(t["amp"], WINDOW, STRIDE)
    if len(w) == 0:
        print(f"  [skip] trial {t['trial_id']} ngắn hơn {WINDOW} gói")
        continue
    X_list.append(w)
    y_list.extend([present_labels.index(t["label"])] * len(w))
    group_list.extend([t["trial_id"]] * len(w))
    trial_list.extend([ti] * len(w))

if not X_list:
    raise SystemExit("Không tạo được cửa sổ nào. Cần trial >= WINDOW gói.")

X = np.concatenate(X_list, axis=0)
y = np.asarray(y_list, dtype=np.int32)
groups = np.asarray(group_list)

print("Tổng cửa sổ:", len(X), "shape:", X.shape)
print("Nhãn có mặt:", present_labels)
for i, name in enumerate(present_labels):
    print(f"  {name:6s}: {int(np.sum(y == i))} windows")

if "fall" not in present_labels:
    print("\n*** CẢNH BÁO: KHÔNG CÓ NHÃN fall -> model KHÔNG THỂ phát hiện ngã. ***")
if len(present_labels) < len(LABELS):
    print("*** CẢNH BÁO: thiếu nhãn:", sorted(set(LABELS) - set(present_labels)), "***")

# ---------------------------------------------------------------------------
# Split: ưu tiên theo trial; nếu không đủ thì split theo thời gian
# ---------------------------------------------------------------------------
n_trials_per_label = {l: len({t["trial_id"] for t in trial_data if t["label"] == l}) for l in present_labels}
print("Số trial / nhãn:", n_trials_per_label)

use_group_split = all(v >= 2 for v in n_trials_per_label.values()) and len(set(groups)) >= 3
if use_group_split:
    from sklearn.model_selection import GroupShuffleSplit
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
    train_idx, test_idx = next(gss.split(X, y, groups))
    train_groups = set(groups[train_idx])
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    tr_idx, val_idx = next(gss2.split(X[train_idx], y[train_idx], groups[train_idx]))
    tr_idx, val_idx = train_idx[tr_idx], train_idx[val_idx]
    split_name = "group_by_trial"
else:
    # Chronological split trong từng trial: 80% đầu train, 20% cuối test
    tr_idx, te_idx = [], []
    for ti in sorted(set(trial_list)):
        idx = np.where(np.asarray(trial_list) == ti)[0]
        if len(idx) < 2:
            tr_idx.extend(idx.tolist())
            continue
        n_tr = max(1, int(round(len(idx) * 0.8)))
        tr_idx.extend(idx[:n_tr].tolist())
        te_idx.extend(idx[n_tr:].tolist())
    tr_idx = np.asarray(tr_idx, dtype=np.int32)
    te_idx = np.asarray(te_idx, dtype=np.int32)
    if len(te_idx) == 0:
        # quá ít window: dùng 1 window làm test
        tr_idx = np.arange(max(1, len(X) - 1))
        te_idx = np.arange(max(1, len(X) - 1), len(X))
    # val lấy 20% từ train
    rng = np.random.default_rng(SEED)
    if len(tr_idx) >= 5:
        perm = rng.permutation(len(tr_idx))
        n_val = max(1, int(round(len(tr_idx) * 0.2)))
        val_idx = tr_idx[perm[:n_val]]
        tr_idx = tr_idx[perm[n_val:]]
    else:
        val_idx = tr_idx[-1:]
        tr_idx = tr_idx[:-1] if len(tr_idx) > 1 else tr_idx
    split_name = "chronological_within_trial"
    print("\n*** CẢNH BÁO: không đủ 2 trial/nhãn -> dùng split theo thời gian.")
    print("*** Kết quả sẽ lạc quan hơn thực tế; cần thu thêm trial/người/phòng. ***")

X_train, y_train = X[tr_idx], y[tr_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[te_idx], y[te_idx]
print(f"\nSplit: {split_name}")
print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)


In [ ]:
# ---------------------------------------------------------------------------
# Chuẩn hoá per-window z-score (+ PCA tuỳ chọn)
# ---------------------------------------------------------------------------
def zscore_windows(x):
    mu = x.mean(axis=1, keepdims=True)
    sd = x.std(axis=1, keepdims=True) + 1e-6
    return (x - mu) / sd


X_train = zscore_windows(X_train).astype(np.float32)
X_val = zscore_windows(X_val).astype(np.float32)
X_test = zscore_windows(X_test).astype(np.float32)

pca_mean = pca_components = None
if USE_PCA:
    from sklearn.decomposition import PCA
    n_flat = X_train.shape[1]
    n_feat = X_train.shape[2]
    pca = PCA(n_components=min(PCA_COMPONENTS, n_feat), random_state=SEED)
    pca.fit(X_train.reshape(-1, n_feat))
    def apply_pca(x):
        return pca.transform(x.reshape(-1, n_feat)).reshape(x.shape[0], n_flat, -1).astype(np.float32)
    X_train = apply_pca(X_train)
    X_val = apply_pca(X_val)
    X_test = apply_pca(X_test)
    pca_mean = pca.mean_.astype(float).tolist()
    pca_components = pca.components_.astype(float).tolist()
    print("PCA:", pca.explained_variance_ratio_.sum() * 100, "% variance")

input_shape = X_train.shape[1:]
n_classes = len(present_labels)
print("input_shape:", input_shape, "n_classes:", n_classes)


In [ ]:
# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
def build_cnn_small(shape, n_classes):
    inputs = tf.keras.Input(shape=shape, name="csi")
    x = tf.keras.layers.Conv1D(16, 5, padding="same", activation="relu")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Conv1D(32, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Conv1D(48, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(32, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="csi_cnn_small")


def build_cnn_lstm(shape, n_classes):
    inputs = tf.keras.Input(shape=shape, name="csi")
    x = tf.keras.layers.Conv1D(32, 3, activation="relu")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Conv1D(64, 3, activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.LSTM(128, return_sequences=True)(x)
    x = tf.keras.layers.LSTM(64)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="csi_cnn_lstm")


if MODEL_VARIANT == "cnn_lstm":
    model = build_cnn_lstm(input_shape, n_classes)
else:
    model = build_cnn_small(input_shape, n_classes)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()


In [ ]:
# ---------------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)

classes = np.unique(y_train)
cw = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(classes, cw)}
print("class_weight:", class_weight)

batch_size = int(min(32, max(1, len(X_train) // 4)))
print("batch_size:", batch_size)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=12,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(str(OUT_DIR / "best_model.keras"),
                                       monitor="val_accuracy", save_best_only=True,
                                       verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                         patience=5, min_lr=1e-5, verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=batch_size,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=2,
)

pd.DataFrame(history.history).to_csv(OUT_DIR / "training_history.csv", index=False)


In [ ]:
# ---------------------------------------------------------------------------
# Evaluate
# ---------------------------------------------------------------------------
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
acc = float(np.mean(y_pred == y_test)) if len(y_test) else float("nan")
macro_f1 = float(f1_score(y_test, y_pred, average="macro", zero_division=0)) if len(y_test) else float("nan")

report = classification_report(
    y_test, y_pred,
    labels=list(range(n_classes)),
    target_names=present_labels,
    zero_division=0,
)
print("Test accuracy:", acc)
print("Macro F1:", macro_f1)
print("\nClassification Report:")
print(report)

cm = confusion_matrix(y_test, y_pred, labels=list(range(n_classes)))
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=present_labels, yticklabels=present_labels)
plt.title(f"Confusion Matrix - acc={acc:.3f}, macroF1={macro_f1:.3f}")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix.png", dpi=160)
plt.close()

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history.get("loss", []), label="train")
axes[0].plot(history.history.get("val_loss", []), label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(history.history.get("accuracy", []), label="train")
axes[1].plot(history.history.get("val_accuracy", []), label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "training_curves.png", dpi=160)
plt.close()

# FAR (xấp xỉ) nếu có empty và fall
median_pps = float(np.nanmedian([t["pps"] for t in trial_data])) if trial_data else float("nan")
if not np.isfinite(median_pps) or median_pps <= 0:
    median_pps = 75.0
window_seconds = WINDOW / median_pps
far_per_hour = None
fall_recall = None

if "empty" in present_labels and "fall" in present_labels:
    ei = present_labels.index("empty")
    fi = present_labels.index("fall")
    n_empty = int(np.sum(y_test == ei))
    n_fa = int(np.sum((y_test == ei) & (y_pred == fi)))
    far_per_window = n_fa / max(1, n_empty)
    far_per_hour = far_per_window * (3600.0 / window_seconds)

if "fall" in present_labels and cm.shape[0] > present_labels.index("fall"):
    fi = present_labels.index("fall")
    fall_recall = float(cm[fi, fi] / max(1, cm[fi].sum()))
    fall_precision = float(cm[fi, fi] / max(1, cm[:, fi].sum()))
else:
    fall_precision = None

print("median_pps:", median_pps, "window_seconds:", window_seconds)
print("fall_recall:", fall_recall, "fall_precision:", fall_precision, "FAR/hour (approx):", far_per_hour)


In [ ]:
# ---------------------------------------------------------------------------
# TFLite export (int8 cho ESP32-S3) + model_data.h
# ---------------------------------------------------------------------------
def build_int8_tflite(keras_model, x_train):
    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    def rep():
        n = min(len(x_train), 300)
        for i in range(n):
            yield [x_train[i:i + 1].astype(np.float32)]

    converter.representative_dataset = rep
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    return converter.convert()


def tflite_predict(model_bytes, x):
    interpreter = tf.lite.Interpreter(model_content=model_bytes)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    in_scale, in_zero = inp["quantization"]
    out_scale, out_zero = out["quantization"]
    preds = []
    for i in range(len(x)):
        xi = x[i:i + 1]
        if inp["dtype"] == np.int8:
            xi = np.clip(np.round(xi / in_scale + in_zero), -128, 127).astype(np.int8)
        else:
            xi = xi.astype(np.float32)
        interpreter.set_tensor(inp["index"], xi)
        interpreter.invoke()
        o = interpreter.get_tensor(out["index"])[0]
        if out["dtype"] == np.int8:
            o = (o.astype(np.float32) - out_zero) * out_scale
        preds.append(int(np.argmax(o)))
    return np.asarray(preds, dtype=np.int32), inp, out


model.save(OUT_DIR / "model.keras")

float_converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_float = float_converter.convert()
(OUT_DIR / "model_float.tflite").write_bytes(tflite_float)

tflite_int8 = build_int8_tflite(model, X_train)
(OUT_DIR / "model.tflite").write_bytes(tflite_int8)

tflite_acc = float("nan")
inp_details = out_details = None
if len(X_test):
    tpred, inp_details, out_details = tflite_predict(tflite_int8, X_test)
    tflite_acc = float(np.mean(tpred == y_test))
    print("TFLite int8 accuracy:", tflite_acc, "drop:", acc - tflite_acc)

# model_data.h
def to_c_header(data: bytes, name="model_data"):
    hexes = [f"0x{b:02x}" for b in data]
    rows = []
    for i in range(0, len(hexes), 12):
        rows.append("  " + ", ".join(hexes[i:i + 12]) + ",")
    return (
        "// Auto-generated by ai/notebooks/train_esp32s3_6class.ipynb\n"
        "#pragma once\n#include <stddef.h>\n"
        f"__attribute__((aligned(8)))\nconst unsigned char {name}[] = {{\n"
        + "\n".join(rows) + "\n};\n"
        f"const unsigned int {name}_len = {len(data)};\n"
    )

(OUT_DIR / "model_data.h").write_text(to_c_header(tflite_int8), encoding="utf-8")

# labels + model_info
(OUT_DIR / "labels.json").write_text(json.dumps(present_labels, indent=2), encoding="utf-8")

in_scale = float(inp_details["quantization"][0]) if inp_details is not None else None
in_zero = int(inp_details["quantization"][1]) if inp_details is not None else None
out_scale = float(out_details["quantization"][0]) if out_details is not None else None
out_zero = int(out_details["quantization"][1]) if out_details is not None else None

deployable = (
    set(present_labels) == set(LABELS)
    and fall_recall is not None and fall_recall >= 0.90
    and macro_f1 >= 0.85
    and len(X) >= 200
    and all(v >= 2 for v in n_trials_per_label.values())
)

info = {
    "model_variant": MODEL_VARIANT,
    "input_shape": [int(v) for v in model.input_shape[1:]],
    "window": WINDOW,
    "stride": STRIDE,
    "n_subcarriers": int(X.shape[2]),
    "normalization": "per_window_zscore",
    "use_pca": bool(USE_PCA),
    "pca_components": int(PCA_COMPONENTS) if USE_PCA else 0,
    "pca_mean": pca_mean,
    "pca_components_matrix": pca_components,
    "labels": present_labels,
    "n_classes": int(n_classes),
    "tflite_int8": bool(TFLITE_INT8),
    "input_quant": {"scale": in_scale, "zero_point": in_zero, "dtype": "int8"},
    "output_quant": {"scale": out_scale, "zero_point": out_zero, "dtype": "int8"},
    "keras_test_acc": acc,
    "macro_f1": macro_f1,
    "tflite_int8_test_acc": tflite_acc,
    "fall_recall": fall_recall,
    "fall_precision": fall_precision,
    "far_per_hour_approx": far_per_hour,
    "median_pps": median_pps,
    "n_windows": int(len(X)),
    "n_trials": int(len(trial_data)),
    "n_trials_per_label": {k: int(v) for k, v in n_trials_per_label.items()},
    "split": split_name,
    "deployable": bool(deployable),
    "deploy_notes": (
        "OK" if deployable else
        "Chưa đủ điều kiện deploy: cần đủ 6 nhãn, fall_recall>=0.90, macroF1>=0.85, "
        ">=200 windows, >=2 trial/nhãn."
    ),
}
(OUT_DIR / "model_info.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
(OUT_DIR / "metrics.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
(OUT_DIR / "classification_report.txt").write_text(report, encoding="utf-8")

print("\nDEPLOYABLE:", deployable)
print(info["deploy_notes"])


In [ ]:
# ---------------------------------------------------------------------------
# model_card.md + zip
# ---------------------------------------------------------------------------
card = f"""# Model card - ESP32-S3 CSI 6 classes

- Notebook: `ai/notebooks/train_esp32s3_6class.ipynb`
- Variant: `{MODEL_VARIANT}`
- Input: `{info['input_shape']}` (window={WINDOW}, subcarriers={info['n_subcarriers']})
- Normalization: per-window z-score
- PCA: {USE_PCA} ({PCA_COMPONENTS if USE_PCA else 0} components)
- Labels: {present_labels}
- Split: {split_name}
- Windows: {len(X)} (train={len(X_train)}, val={len(X_val)}, test={len(X_test)})
- Trials: {len(trial_data)}

## Metrics
- Keras test accuracy: {acc:.4f}
- Macro F1: {macro_f1:.4f}
- TFLite int8 accuracy: {tflite_acc:.4f}
- Fall recall: {fall_recall}
- Fall precision: {fall_precision}
- FAR/hour (approx): {far_per_hour}
- Deployable: **{deployable}**

## Limitations
- Dataset provenance: CSI collected with `csi_labeler` on ESP32-S3 (384 int8 = 192 I/Q).
- If `fall` is missing or few trials, the model is a pipeline smoke test only.
- Do not claim real-world fall performance from this card unless `deployable=true`.
"""
(OUT_DIR / "model_card.md").write_text(card, encoding="utf-8")

zip_path = shutil.make_archive("/kaggle/working/esp32s3_csi_model", "zip", OUT_DIR)
print("Artifacts:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name:24s} {p.stat().st_size:>10d} bytes")
print("\nZIP:", zip_path)
print("Hãy tải file ZIP này và gửi lại để được đánh giá.")


## Gửi lại gì?

Tải file `/kaggle/working/esp32s3_csi_model.zip` và gửi lại. Tối thiểu cần:

- `metrics.json` / `model_info.json`
- `classification_report.txt`
- `confusion_matrix.png`
- `training_curves.png`
- `model_card.md`

Nếu muốn mình tích hợp firmware, gửi thêm `model.tflite` + `model_data.h` (đã có trong zip).

## Tiêu chí để model được dùng

- Fall recall >= 0.90, precision >= 0.85
- FAR <= 1 báo động giả/giờ trên `empty`
- Macro F1 >= 0.85
- TFLite int8 giảm <= 2% so với Keras
- Test trên trial/người/phòng held-out
- Đủ 6 nhãn, >= 200 windows

Nếu chưa đạt, notebook sẽ ghi `deployable=false`; khi đó chỉ dùng rule-based fallback và thu thêm dữ liệu.
